In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 10


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 2.7337168753147125
Epoch 2/100, Loss: 2.7302338406443596
Epoch 3/100, Loss: 3.0357079580426216
Epoch 4/100, Loss: 3.2690871357917786
Epoch 5/100, Loss: 2.5035408437252045
Epoch 6/100, Loss: 3.2735675126314163
Epoch 7/100, Loss: 2.5745821744203568
Epoch 8/100, Loss: 2.559894844889641
Epoch 9/100, Loss: 2.7290377616882324
Epoch 10/100, Loss: 2.8923652544617653
Epoch 11/100, Loss: 3.077162116765976
Epoch 12/100, Loss: 2.746838830411434
Epoch 13/100, Loss: 2.92381202429533
Epoch 14/100, Loss: 2.4565949514508247
Epoch 15/100, Loss: 2.4977508261799812
Epoch 16/100, Loss: 2.5896700844168663
Epoch 17/100, Loss: 3.6647697910666466


Epoch 18/100, Loss: 2.538254290819168
Epoch 19/100, Loss: 3.1665839180350304
Epoch 20/100, Loss: 2.621709242463112
Epoch 21/100, Loss: 2.800230532884598
Epoch 22/100, Loss: 2.862464725971222
Epoch 23/100, Loss: 2.801221713423729
Epoch 24/100, Loss: 3.088426485657692
Epoch 25/100, Loss: 2.6065587624907494
Epoch 26/100, Loss: 3.438803344964981
Epoch 27/100, Loss: 2.7559598609805107
Epoch 28/100, Loss: 2.810886934399605
Epoch 29/100, Loss: 2.7731908708810806
Epoch 30/100, Loss: 3.274043545126915


Epoch 31/100, Loss: 2.803301766514778
Epoch 32/100, Loss: 2.747450463473797
Epoch 33/100, Loss: 3.156467705965042
Epoch 34/100, Loss: 2.8124116882681847
Epoch 35/100, Loss: 2.6648919880390167
Epoch 36/100, Loss: 3.0633168518543243
Epoch 37/100, Loss: 3.2623305320739746
Epoch 38/100, Loss: 2.563201457262039
Epoch 39/100, Loss: 2.736765719950199
Epoch 40/100, Loss: 2.752391129732132
Epoch 41/100, Loss: 3.106877364218235
Epoch 42/100, Loss: 2.8098677694797516
Epoch 43/100, Loss: 2.9984136894345284
Epoch 44/100, Loss: 3.0638600140810013
Epoch 45/100, Loss: 3.186799369752407
Epoch 46/100, Loss: 2.9063624814152718


Epoch 47/100, Loss: 2.4907381758093834
Epoch 48/100, Loss: 2.8318695202469826
Epoch 49/100, Loss: 3.312658116221428
Epoch 50/100, Loss: 2.6203112974762917
Epoch 51/100, Loss: 2.7435004338622093
Epoch 52/100, Loss: 2.629369743168354
Epoch 53/100, Loss: 2.9081513956189156
Epoch 54/100, Loss: 2.60598386824131
Epoch 55/100, Loss: 2.5275545865297318
Epoch 56/100, Loss: 2.789253629744053
Epoch 57/100, Loss: 3.126452349126339
Epoch 58/100, Loss: 3.1295663863420486
Epoch 59/100, Loss: 3.1253997907042503
Epoch 60/100, Loss: 3.2813093215227127
Epoch 61/100, Loss: 2.840199537575245
Epoch 62/100, Loss: 2.8054833188652992
Epoch 63/100, Loss: 2.7908602505922318
Epoch 64/100, Loss: 2.680377572774887


Epoch 65/100, Loss: 2.6463449969887733
Epoch 66/100, Loss: 2.971904829144478
Epoch 67/100, Loss: 2.843291997909546
Epoch 68/100, Loss: 2.919420689344406
Epoch 69/100, Loss: 3.1346609070897102
Epoch 70/100, Loss: 2.9815763980150223
Epoch 71/100, Loss: 2.707494981586933
Epoch 72/100, Loss: 2.8484889194369316
Epoch 73/100, Loss: 2.5308615416288376
Epoch 74/100, Loss: 3.008345253765583
Epoch 75/100, Loss: 2.7854294031858444
Epoch 76/100, Loss: 2.540107846260071
Epoch 77/100, Loss: 2.468180052936077
Epoch 78/100, Loss: 3.1235897466540337
Epoch 79/100, Loss: 3.234384000301361
Epoch 80/100, Loss: 2.45659851282835
Epoch 81/100, Loss: 2.882581204175949
Epoch 82/100, Loss: 3.115098938345909


...

Epoch 36/100, Loss: 1.9863765388727188
Epoch 37/100, Loss: 1.9087652489542961
Epoch 38/100, Loss: 2.1406190171837807
Epoch 39/100, Loss: 2.0112396627664566
Epoch 40/100, Loss: 2.1081808358430862
Epoch 41/100, Loss: 2.059292145073414
Epoch 42/100, Loss: 1.961548000574112
Epoch 43/100, Loss: 2.050445109605789
Epoch 44/100, Loss: 1.9779767468571663
Epoch 45/100, Loss: 1.9014254808425903
Epoch 46/100, Loss: 2.084978297352791
Epoch 47/100, Loss: 1.9245680533349514
Epoch 48/100, Loss: 1.9117961302399635
Epoch 49/100, Loss: 2.075766757130623
Epoch 50/100, Loss: 1.9089673683047295
Epoch 51/100, Loss: 1.8396358042955399
Epoch 52/100, Loss: 1.8857063576579094


Epoch 53/100, Loss: 1.9669331200420856
Epoch 54/100, Loss: 1.9659971594810486
Epoch 55/100, Loss: 1.9513743966817856
Epoch 56/100, Loss: 2.00663585960865
Epoch 57/100, Loss: 2.0324851870536804
Epoch 58/100, Loss: 2.1050940454006195
Epoch 59/100, Loss: 2.0546351447701454
Epoch 60/100, Loss: 2.0079597383737564
Epoch 61/100, Loss: 2.0662890672683716
Epoch 62/100, Loss: 1.9432703629136086
Epoch 63/100, Loss: 1.9951411038637161
Epoch 64/100, Loss: 2.061807744204998
Epoch 65/100, Loss: 1.9871901869773865
Epoch 66/100, Loss: 2.008197121322155
Epoch 67/100, Loss: 2.1942691579461098
Epoch 68/100, Loss: 2.038957104086876
Epoch 69/100, Loss: 1.9459349885582924


Epoch 70/100, Loss: 2.0692763328552246
Epoch 71/100, Loss: 1.9954404905438423
Epoch 72/100, Loss: 1.928309105336666
Epoch 73/100, Loss: 1.95933049172163
Epoch 74/100, Loss: 1.90286935120821
Epoch 75/100, Loss: 1.9840053543448448
Epoch 76/100, Loss: 1.9610862657427788
Epoch 77/100, Loss: 2.186746396124363
Epoch 78/100, Loss: 1.980754777789116
Epoch 79/100, Loss: 1.885779395699501
Epoch 80/100, Loss: 1.9037374258041382
Epoch 81/100, Loss: 1.990569956600666
Epoch 82/100, Loss: 1.9498741999268532
Epoch 83/100, Loss: 2.0263502299785614
Epoch 84/100, Loss: 1.8848160654306412
Epoch 85/100, Loss: 2.0705652460455894
Epoch 86/100, Loss: 2.0774721279740334


Epoch 87/100, Loss: 2.0138765200972557
Epoch 88/100, Loss: 2.071864202618599
Epoch 89/100, Loss: 2.0125782266259193
Epoch 90/100, Loss: 1.970723681151867
Epoch 91/100, Loss: 2.218948796391487
Epoch 92/100, Loss: 2.0287037566304207
Epoch 93/100, Loss: 1.8996108993887901
Epoch 94/100, Loss: 2.0644555389881134
Epoch 95/100, Loss: 2.0044837221503258
Epoch 96/100, Loss: 2.1798882633447647
Epoch 97/100, Loss: 1.9341654255986214
Epoch 98/100, Loss: 2.006088934838772


Epoch 99/100, Loss: 2.1258654445409775
Epoch 100/100, Loss: 2.0418042838573456
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.7476
